# DeepDetect Experiments V2: convnext_base
V2 pipeline: v5 skin-tone patch + phase FFT + original cleaner + dual transforms.
Spatial branch: full degradation augmentations.
Frequency branch: spatial aug only — patch selected from clean image.

Best hyperparameters from Optuna tuning (trial 13):
  backbone_lr=4.85e-05, lr=2.79e-04, freq_aux_weight=0.500,
  diversity_weight=0.058, gate_init_bias=0.215


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
VRAM: 25.2 GB


## Best Hyperparameters

In [2]:
BEST_BACKBONE_LR       = 4.85e-05
BEST_LR                = 2.79e-04
BEST_FREQ_AUX_WEIGHT   = 0.500
BEST_DIVERSITY_WEIGHT  = 0.058
BEST_GATE_INIT_BIAS    = 0.215


## Shared Config

In [3]:
from config import Config
from data.deepdetect_dual import get_deepdetect_dual_loaders
from experiments.train import train_v2
from experiments.evaluate import full_evaluation_v2
from experiments.baseline_spatial_only import run_spatial_only_baseline

BACKBONE = "convnext_base"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.deepdetect_root      = "../data/raw/deep_detect/data"
    cfg.data.dataset              = "deepdetect"
    cfg.data.image_size           = 224
    cfg.data.batch_size           = 88
    cfg.data.num_workers          = 4
    cfg.backbone.name             = BACKBONE
    cfg.backbone.pretrained       = True
    cfg.backbone.img_size         = 224
    cfg.backbone.frozen           = frozen
    cfg.fusion.mode               = fusion_mode
    cfg.frequency.use_v2_pipeline = True
    cfg.frequency.cleaner_filters = 3
    cfg.frequency.patch_size      = 56
    cfg.train.early_stopping_patience = 30  
    cfg.train.epochs              = 35
    cfg.train.backbone_lr         = BEST_BACKBONE_LR
    cfg.train.lr                  = BEST_LR
    cfg.loss.freq_aux_weight      = BEST_FREQ_AUX_WEIGHT
    cfg.fusion.diversity_weight   = BEST_DIVERSITY_WEIGHT
    cfg.fusion.gate_init_bias     = BEST_GATE_INIT_BIAS
    # Spatial branch — full augmentations
    cfg.data.jpeg_aug             = True
    cfg.data.blur_aug             = True
    cfg.data.noise_aug            = True
    cfg.data.recompression_aug    = True
    cfg.data.mixed_aug            = True
    cfg.data.mixed_aug_prob       = 0.3
    cfg.experiment_name = f"dd_v2_{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes           = f"DeepDetect V2 · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

cfg_data = make_cfg("joint_only")
train_loader, val_loader, test_loader = get_deepdetect_dual_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776


## Experiment 0: Spatial-Only Baseline

In [ ]:
cfg0 = make_cfg("joint_only")
cfg0.experiment_name = f"dd_v2_{BACKBONE}_spatial_only"
cfg0.notes           = f"DeepDetect V2 spatial-only baseline"
cfg0.train.epochs    = 20
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")


Device: cuda
Training spatial-only baseline (convnext_base) for 20 epochs...
Train: 76,848  Val: 13,561


Epoch   1/20 | train_loss=0.1279 | val_acc=99.1%


Epoch   2/20 | train_loss=0.0426 | val_acc=98.5%


Epoch   3/20 | train_loss=0.0264 | val_acc=98.9%


Epoch   4/20 | train_loss=0.0214 | val_acc=99.7%


...

Epoch  16/20 | train_loss=0.0010 | val_acc=99.7%


Epoch  17/20 | train_loss=0.0010 | val_acc=99.7%


Epoch  18/20 | train_loss=0.0007 | val_acc=99.9%


Epoch  19/20 | train_loss=0.0005 | val_acc=99.8%


Epoch  20/20 | train_loss=0.0001 | val_acc=99.9%

Spatial-only results (convnext_base):
  Accuracy: 96.1%
  AUC-ROC:  0.990
  F1:       0.961
Results saved → ./results/results.csv  (dd_v2_convnext_base_spatial_only, acc=0.9611)
Results saved to ./results/results.csv

Spatial-only floor: 96.1%


## Experiment 1: Joint-Only

In [4]:
cfg1 = make_cfg("joint_only")
train_v2(cfg1, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_convnext_base_joint_only [V2 pipeline]
Backbone: convnext_base | Fusion: joint_only | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5201 | val_acc=96.0% | val_auc=1.000 | val_f1=0.954 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=96.0%)


Epoch   2/35 | train_loss=0.3905 | val_acc=99.6% | val_auc=1.000 | val_f1=0.995 | best=96.0% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=99.6%)


Epoch   3/35 | train_loss=0.3621 | val_acc=98.8% | val_auc=1.000 | val_f1=0.987 | best=99.6% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=98.8%)


Epoch   4/35 | train_loss=0.3409 | val_acc=99.6% | val_auc=1.000 | val_f1=0.995 | best=99.6% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=99.6%)


...

Epoch  32/35 | train_loss=0.2544 | val_acc=99.8% | val_auc=1.000 | val_f1=0.998 | best=99.9% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.8%)


Epoch  33/35 | train_loss=0.2537 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.9% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.7%)


Epoch  34/35 | train_loss=0.2540 | val_acc=99.8% | val_auc=1.000 | val_f1=0.998 | best=99.9% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.8%)


Epoch  35/35 | train_loss=0.2546 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.9% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.7%)

Training complete. Best val accuracy: 99.9%
Results will be logged to CSV after full_evaluation_v2().


0.9986726642577981

In [5]:
results1 = full_evaluation_v2(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)

Loaded checkpoint: ./checkpoints/best_dd_v2_convnext_base_joint_only.pt

EVALUATION — dd_v2_convnext_base_joint_only
Backbone: convnext_base | Fusion: joint_only | Frozen: False
  Joint accuracy:     97.2%
  AUC-ROC:            0.985
  F1:                 0.971
  Spatial-only:       97.2%
  Freq-only:          73.4%
  Delta (Δ):          +0.0%  (freq branch contribution)

  No warning signs triggered.
Results saved → ./results/results.csv  (dd_v2_convnext_base_joint_only, acc=0.9718)



## Experiment 2: Scalar Fusion

In [6]:
cfg2 = make_cfg("scalar")
train_v2(cfg2, train_loader, val_loader, test_loader)

Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_convnext_base_scalar [V2 pipeline]
Backbone: convnext_base | Fusion: scalar | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5140 | val_acc=98.2% | val_auc=1.000 | val_f1=0.980 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=98.2%)


Epoch   2/35 | train_loss=0.3890 | val_acc=97.8% | val_auc=1.000 | val_f1=0.976 | best=98.2% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=97.8%)


Epoch   3/35 | train_loss=0.3605 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=98.2% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=99.7%)


Epoch   4/35 | train_loss=0.3442 | val_acc=99.8% | val_auc=1.000 | val_f1=0.998 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=99.8%)


...

Epoch  32/35 | train_loss=0.2578 | val_acc=99.8% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.8%)


Epoch  33/35 | train_loss=0.2575 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.7%)


Epoch  34/35 | train_loss=0.2574 | val_acc=99.8% | val_auc=1.000 | val_f1=0.998 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.8%)


Epoch  35/35 | train_loss=0.2562 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.7%)

Training complete. Best val accuracy: 99.8%
Results will be logged to CSV after full_evaluation_v2().


0.9981564781358306

In [7]:
results2 = full_evaluation_v2(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)

Loaded checkpoint: ./checkpoints/best_dd_v2_convnext_base_scalar.pt

EVALUATION — dd_v2_convnext_base_scalar
Backbone: convnext_base | Fusion: scalar | Frozen: False
  Joint accuracy:     96.9%
  AUC-ROC:            0.988
  F1:                 0.969
  Spatial-only:       97.0%
  Freq-only:          71.9%
  Delta (Δ):          -0.1%  (freq branch contribution)
Results saved → ./results/results.csv  (dd_v2_convnext_base_scalar, acc=0.9692)



## Experiment 3: Gating Fusion

In [8]:
cfg3 = make_cfg("gating")
train_v2(cfg3, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_convnext_base_gating [V2 pipeline]
Backbone: convnext_base | Fusion: gating | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5447 | val_acc=96.8% | val_auc=1.000 | val_f1=0.964 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=96.8%)


Epoch   2/35 | train_loss=0.4084 | val_acc=97.9% | val_auc=1.000 | val_f1=0.977 | best=96.8% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=97.9%)


Epoch   3/35 | train_loss=0.3765 | val_acc=99.1% | val_auc=1.000 | val_f1=0.990 | best=97.9% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=99.1%)


Epoch   4/35 | train_loss=0.3615 | val_acc=98.2% | val_auc=1.000 | val_f1=0.980 | best=99.1% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=98.2%)


...

Epoch  32/35 | train_loss=0.2578 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.9% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.7%)


Epoch  33/35 | train_loss=0.2583 | val_acc=99.9% | val_auc=1.000 | val_f1=0.999 | best=99.9% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.9%)


Epoch  34/35 | train_loss=0.2596 | val_acc=99.9% | val_auc=1.000 | val_f1=0.998 | best=99.9% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.9%)


Epoch  35/35 | train_loss=0.2575 | val_acc=99.9% | val_auc=1.000 | val_f1=0.999 | best=99.9% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.9%)

Training complete. Best val accuracy: 99.9%
Results will be logged to CSV after full_evaluation_v2().


0.9991151095051988

In [9]:
results3 = full_evaluation_v2(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_v2_convnext_base_gating.pt

EVALUATION — dd_v2_convnext_base_gating
Backbone: convnext_base | Fusion: gating | Frozen: False
  Joint accuracy:     96.5%
  AUC-ROC:            0.988
  F1:                 0.964
  Spatial-only:       96.6%
  Freq-only:          72.5%
  Delta (Δ):          -0.1%  (freq branch contribution)

  Gate distribution:
    entropy:  2.179 nats  (OK)
    mean:     0.527
    variance: 0.0597
Results saved → ./results/results.csv  (dd_v2_convnext_base_gating, acc=0.9645)



## Experiment 4: Gating Frozen

In [10]:
cfg4 = make_cfg("gating", frozen=True)
train_v2(cfg4, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_convnext_base_gating_frozen [V2 pipeline]
Backbone: convnext_base | Fusion: gating | Frozen: True
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.8113 | val_acc=89.5% | val_auc=0.967 | val_f1=0.879 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=89.5%)


Epoch   2/35 | train_loss=0.6908 | val_acc=88.0% | val_auc=0.976 | val_f1=0.854 | best=89.5% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=88.0%)


Epoch   3/35 | train_loss=0.6513 | val_acc=91.4% | val_auc=0.980 | val_f1=0.902 | best=89.5% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=91.4%)


Epoch   4/35 | train_loss=0.6144 | val_acc=88.9% | val_auc=0.982 | val_f1=0.865 | best=91.4% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=88.9%)


...

Epoch  32/35 | train_loss=0.3946 | val_acc=93.8% | val_auc=0.993 | val_f1=0.928 | best=95.2% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=93.8%)


Epoch  33/35 | train_loss=0.3941 | val_acc=94.5% | val_auc=0.993 | val_f1=0.938 | best=95.2% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=94.5%)


Epoch  34/35 | train_loss=0.3889 | val_acc=94.4% | val_auc=0.993 | val_f1=0.936 | best=95.2% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=94.4%)


Epoch  35/35 | train_loss=0.3914 | val_acc=94.4% | val_auc=0.993 | val_f1=0.936 | best=95.2% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=94.4%)

Training complete. Best val accuracy: 95.2%
Results will be logged to CSV after full_evaluation_v2().


0.9515522454096306

In [11]:
results4 = full_evaluation_v2(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_v2_convnext_base_gating_frozen.pt

EVALUATION — dd_v2_convnext_base_gating_frozen
Backbone: convnext_base | Fusion: gating | Frozen: True
  Joint accuracy:     90.6%
  AUC-ROC:            0.967
  F1:                 0.900
  Spatial-only:       86.8%
  Freq-only:          72.0%
  Delta (Δ):          +3.8%  (freq branch contribution)

  Gate distribution:
    entropy:  2.949 nats  (OK)
    mean:     0.532
    variance: 0.0836

  No warning signs triggered.
Results saved → ./results/results.csv  (dd_v2_convnext_base_gating_frozen, acc=0.9061)



## Summary

In [16]:
import pandas as pd
df = pd.read_csv("./results/results.csv", on_bad_lines='skip')
mask = df["experiment_name"].str.startswith(f"dd_v2_{BACKBONE}")
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))


                  experiment_name fusion  frozen  accuracy  auc_roc     f1  gate_entropy
       dd_v2_convnext_base_scalar scalar   False    0.9692   0.9885 0.9687        0.0000
       dd_v2_convnext_base_gating gating   False    0.9645   0.9879 0.9641        2.1788
dd_v2_convnext_base_gating_frozen gating    True    0.9061   0.9669 0.9000        2.9492
